# EDA for U.S. Inflation Forecasting

**Topic:** Forecasting U.S. Inflation Using Time Series and Machine Learning Models: The Role of Unemployment

This notebook prepares and explores the monthly U.S. macroeconomic dataset used in the report.

Main tasks:

1. Load and merge CPI, unemployment, federal funds rate, and industrial production data.
2. Restrict the sample to **January 1990 – April 2026**.
3. Handle missing observations if any remain.
4. Construct year-over-year inflation and industrial production growth.
5. Produce descriptive statistics and exploratory plots.
6. Create train/test split information for later modeling.


## 1. Import libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True

## 2. Load raw data

The notebook expects the following files to be in the same folder as the notebook. If the notebook is run in ChatGPT's sandbox, it will also search `/mnt/data`.

- `CPIAUCSL_update.csv`
- `UNRATE_update.csv`
- `FEDFUNDS.csv`
- `INDPRO.csv`


In [ ]:
def find_file(filename):
    # Find a file either in the current directory or in /mnt/data.
    candidates = [Path(filename), Path("/mnt/data") / filename]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Put it in the same folder as this notebook."
    )

files = {
    "CPIAUCSL": find_file("CPIAUCSL_update.csv"),
    "UNRATE": find_file("UNRATE_update.csv"),
    "FEDFUNDS": find_file("FEDFUNDS.csv"),
    "INDPRO": find_file("INDPRO.csv"),
}

files

In [ ]:
def load_fred_csv(path, value_col):
    df = pd.read_csv(path)
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df = df.rename(columns={"observation_date": "DATE"})
    df = df[["DATE", value_col]].sort_values("DATE")
    return df

cpi = load_fred_csv(files["CPIAUCSL"], "CPIAUCSL")
unrate = load_fred_csv(files["UNRATE"], "UNRATE")
fedfunds = load_fred_csv(files["FEDFUNDS"], "FEDFUNDS")
indpro = load_fred_csv(files["INDPRO"], "INDPRO")

for name, df_i in {
    "CPIAUCSL": cpi,
    "UNRATE": unrate,
    "FEDFUNDS": fedfunds,
    "INDPRO": indpro,
}.items():
    print(f"{name}: {df_i['DATE'].min().date()} to {df_i['DATE'].max().date()}, n = {len(df_i)}")

## 3. Merge and restrict sample

The report uses the sample from **January 1990 to April 2026**. This provides a modern sample with enough monthly observations for time series models and moderate-complexity machine learning models.


In [ ]:
# Merge all series by monthly observation date
raw = cpi.merge(unrate, on="DATE", how="outer")
raw = raw.merge(fedfunds, on="DATE", how="outer")
raw = raw.merge(indpro, on="DATE", how="outer")
raw = raw.sort_values("DATE").set_index("DATE")

# Restrict sample
start_date = "1990-01-01"
end_date = "2026-04-01"
raw = raw.loc[start_date:end_date].copy()

print(raw.shape)
raw.head()

In [ ]:
raw.tail()

## 4. Check and handle missing values

If isolated one-month missing values remain, linear interpolation is used to preserve the monthly frequency. This is especially important before constructing lag features.


In [ ]:
missing_before = raw.isna().sum()
missing_before

In [ ]:
# Show rows with any missing values
raw[raw.isna().any(axis=1)]

In [ ]:
# Fill isolated missing values using time-based linear interpolation.
# If no missing values exist, this step does not change the data.
data = raw.interpolate(method="time")

missing_after = data.isna().sum()
missing_after

## 5. Construct transformed variables

Target variable:

\[
\pi_t = 100 \times [\log(CPI_t) - \log(CPI_{t-12})]
\]

Industrial production growth:

\[
IPG_t = 100 \times [\log(INDPRO_t) - \log(INDPRO_{t-12})]
\]

The unemployment rate and federal funds rate are kept in levels.


In [ ]:
df = data.copy()

df["INFLATION"] = 100 * (np.log(df["CPIAUCSL"]) - np.log(df["CPIAUCSL"].shift(12)))
df["IP_GROWTH"] = 100 * (np.log(df["INDPRO"]) - np.log(df["INDPRO"].shift(12)))

model_data = df[["INFLATION", "UNRATE", "FEDFUNDS", "IP_GROWTH"]].dropna().copy()

print(model_data.index.min().date(), "to", model_data.index.max().date())
print(model_data.shape)
model_data.head()

## 6. Descriptive statistics

In [ ]:
summary = model_data.describe().T
summary = summary[["count", "mean", "std", "min", "50%", "max"]]
summary = summary.rename(columns={"50%": "median"})
summary

In [ ]:
# Optional: export the summary table for report writing
summary.to_csv('eda_summary_statistics.csv')

## 7. Time series plots

The following plots show the behavior of the transformed variables used in modeling.


In [ ]:
ax = model_data["INFLATION"].plot()
ax.set_title("U.S. Year-over-Year CPI Inflation")
ax.set_xlabel("Date")
ax.set_ylabel("Percent")
plt.show()

In [ ]:
ax = model_data["UNRATE"].plot()
ax.set_title("U.S. Unemployment Rate")
ax.set_xlabel("Date")
ax.set_ylabel("Percent")
plt.show()

In [ ]:
ax = model_data["FEDFUNDS"].plot()
ax.set_title("Effective Federal Funds Rate")
ax.set_xlabel("Date")
ax.set_ylabel("Percent")
plt.show()

In [ ]:
ax = model_data["IP_GROWTH"].plot()
ax.set_title("Year-over-Year Industrial Production Growth")
ax.set_xlabel("Date")
ax.set_ylabel("Percent")
plt.show()

## 8. Inflation and unemployment

This plot is only descriptive. It helps visualize the relationship motivated by the Phillips Curve, but the formal evidence in the report should come from forecasting performance and VAR/Granger tests.


In [ ]:
ax = model_data[["INFLATION", "UNRATE"]].plot()
ax.set_title("Inflation and Unemployment Rate")
ax.set_xlabel("Date")
ax.set_ylabel("Percent")
plt.show()

In [ ]:
ax = model_data.plot.scatter(x="UNRATE", y="INFLATION")
ax.set_title("Scatter Plot: Inflation vs. Unemployment")
ax.set_xlabel("Unemployment Rate (%)")
ax.set_ylabel("YoY CPI Inflation (%)")
plt.show()

## 9. Correlation matrix

Correlation is useful for preliminary exploration, but it should not be interpreted as causal evidence.


In [ ]:
corr = model_data.corr()
corr

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr.values)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

ax.set_title("Correlation Matrix")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 10. Rolling statistics

Rolling statistics provide a simple view of changing inflation behavior over time.


In [ ]:
inflation_roll = pd.DataFrame({
    "Inflation": model_data["INFLATION"],
    "12-month rolling mean": model_data["INFLATION"].rolling(12).mean(),
    "12-month rolling std": model_data["INFLATION"].rolling(12).std(),
})

ax = inflation_roll.plot()
ax.set_title("Inflation with 12-Month Rolling Mean and Standard Deviation")
ax.set_xlabel("Date")
ax.set_ylabel("Percent")
plt.show()

## 11. Train-test split for the report

The report uses a chronological split:

- Training set: January 1990 – December 2018 after transformations begin in January 1991.
- Testing set: January 2019 – April 2026.


In [ ]:
train = model_data.loc[:"2018-12-01"].copy()
test = model_data.loc["2019-01-01":].copy()

print("Train:", train.index.min().date(), "to", train.index.max().date(), "n =", len(train))
print("Test :", test.index.min().date(), "to", test.index.max().date(), "n =", len(test))

## 12. Create lagged features for later ML models

This section is included for convenience. It creates lagged variables from 1 to 12 months. These features can be used later for Ridge Regression and XGBoost.


In [ ]:
def create_lag_features(df, columns, max_lag=12):
    out = df.copy()
    for col in columns:
        for lag in range(1, max_lag + 1):
            out[f"{col}_lag{lag}"] = out[col].shift(lag)
    return out

lag_columns = ["INFLATION", "UNRATE", "FEDFUNDS", "IP_GROWTH"]
ml_data = create_lag_features(model_data, lag_columns, max_lag=12).dropna()

# Target for one-step-ahead forecasting: inflation at time t.
# Features use only lagged values up to t-1.
target = "INFLATION"
features = [col for col in ml_data.columns if "_lag" in col]

print("ML data shape:", ml_data.shape)
print("Number of features:", len(features))
ml_data[[target] + features[:8]].head()

In [ ]:
# Export processed datasets for later modeling
model_data.to_csv("processed_macro_1990_2026.csv")
ml_data.to_csv("processed_macro_lagged_1990_2026.csv")

print("Saved:")
print("- processed_macro_1990_2026.csv")
print("- processed_macro_lagged_1990_2026.csv")
print("- eda_summary_statistics.csv")